# 服务器组成
使用安装和导入将多个 FastMCP 服务器组合成一个更大的应用程序。

随着 MCP 应用程序的增长，您可能需要将工具、资源和提示组织到逻辑模块中，或重用现有的服务器组件。FastMCP 通过两种方法支持组合：

- import_server：用于一次性复制带前缀的组件（静态组合）。
- mount：用于创建主服务器将请求委托给子服务器的实时链接（动态组合）。
​
## 为什么要使用 Compose 服务器？
- 模块化：将大型应用程序分解为更小的、集中的服务器（例如，a WeatherServer、a DatabaseServer、a CalendarServer）。
- 可重用性：创建通用实用程序服务器（例如TextProcessingServer）并将其安装在任何需要的地方。
- 团队合作：不同的团队可以在单独的 FastMCP 服务器上工作，然后再合并。
- 组织：将相关功能按逻辑分组。

## 导入与挂载
导入或安装的选择取决于您的使用情况和要求。

| 特征 | 输入 | 安装 |
| :--- | :--- | :--- |
| 方法 | FastMCP．import＿server（） | FastMCP．mount（） |
| 构图类型 | 一次性副本（静态） | 实时链接（动态） |
| 更新 | 子服务器的更改未反映 | 立即反映对子服务器的更改 |
| 最适合 | 捆绑最终组件 | 模块化运行时组合 |

### 代理服务器
FastMCP 支持MCP 代理，允许您在本地 FastMCP 实例中镜像本地或远程服务器。代理与导入和挂载完全兼容。

### 导入（静态合成）
该import_server()方法将所有组件（工具、资源、模板、提示）从一个FastMCP实例（子服务器）复制到另一个实例（主服务器prefix）。添加A以避免命名冲突。

In [ ]:
from fastmcp import FastMCP
import asyncio

# Define subservers
weather_mcp = FastMCP(name="WeatherService")

@weather_mcp.tool()
def get_forecast(city: str) -> dict:
    """Get weather forecast."""
    return {"city": city, "forecast": "Sunny"}

@weather_mcp.resource("data://cities/supported")
def list_supported_cities() -> list[str]:
    """List cities with weather support."""
    return ["London", "Paris", "Tokyo"]

# Define main server
main_mcp = FastMCP(name="MainApp")

# Import subserver
async def setup():
    await main_mcp.import_server("weather", weather_mcp)

# Result: main_mcp now contains prefixed components:
# - Tool: "weather_get_forecast"
# - Resource: "weather+data://cities/supported" 

if __name__ == "__main__":
    asyncio.run(setup())
    main_mcp.run()

导入的工作原理
当你使用 `await main_mcp.import_server(prefix, subserver)：`

- 工具：来自的所有工具subserver均添加到，main_mcp并使用前缀的名称{prefix}_。
    - subserver.tool(name="my_tool")变成main_mcp.tool(name="{prefix}_my_tool")。
- 资源：所有资源均使用以 为前缀的 URI 添加{prefix}+。
    - subserver.resource(uri="data://info")变成main_mcp.resource(uri="{prefix}+data://info")。
- 资源模板：模板的前缀与资源类似。
    - subserver.resource(uri="data://{id}")变成main_mcp.resource(uri="{prefix}+data://{id}")。
- prompt：所有提示均以类似工具的名称作为前缀。
    - subserver.prompt(name="my_prompt")变成main_mcp.prompt(name="{prefix}_my_prompt")。
    
请注意，import_server会执行组件的一次性复制。导入subserver 后对 所做的更改不会反映在 main_mcp 中。subserver的lifespan上下文也不会由主服务器执行。

## 安装（活连接）
`mount()` 方法会在 `main_mcp` 服务器和子服务器之间创建一个实时链接。运行时，对匹配`prefix`的组件的请求将委托给子服务器，而不是复制组件。

In [ ]:
import asyncio
from fastmcp import FastMCP, Client

# Define subserver
dynamic_mcp = FastMCP(name="DynamicService")

@dynamic_mcp.tool()
def initial_tool():
    """Initial tool demonstration."""
    return "Initial Tool Exists"

# Mount subserver (synchronous operation)
main_mcp = FastMCP(name="MainAppLive")
main_mcp.mount("dynamic", dynamic_mcp)

# Add a tool AFTER mounting - it will be accessible through main_mcp
@dynamic_mcp.tool()
def added_later():
    """Tool added after mounting."""
    return "Tool Added Dynamically!"

# Testing access to mounted tools
async def test_dynamic_mount():
    tools = await main_mcp.get_tools()
    print("Available tools:", list(tools.keys()))
    # Shows: ['dynamic_initial_tool', 'dynamic_added_later']
    
    async with Client(main_mcp) as client:
        result = await client.call_tool("dynamic_added_later")
        print("Result:", result[0].text)
        # Shows: "Tool Added Dynamically!"

if __name__ == "__main__":
    asyncio.run(test_dynamic_mount())

## 安装工作原理

配置安装时：

- Live Link：父服务器与挂载的服务器建立连接。
- 动态更新：通过父级访问时，对已安装服务器的更改会立即反映出来。
- 前缀访问：父服务器使用前缀将请求路由到已挂载的服务器。
- 委托：与前缀匹配的组件请求在运行时被委托给已安装的服务器。

`import_server`适用与命名工具、资源、模板和提示相同的前缀规则。

## 直接安装与代理安装
1. FastMCP 支持两种挂载模式：

- 直接挂载（默认）：父服务器直接访问内存中已挂载服务器的对象。
- 已安装的服务器上未发生任何客户端生命周期事件
- 已安装服务器的生命周期上下文未执行
- 通过直接方法调用来处理通信

2. 代理挂载：父服务器将挂载的服务器视为单独的实体，并通过客户端接口与其通信。
- 完整的客户端生命周期事件发生在已安装的服务器上
- 已安装服务器的生命周期在客户端连接时执行
- 通信通过内存客户端传输进行

In [ ]:
# Direct mounting (default when no custom lifespan)
main_mcp.mount("api", api_server)

# Proxy mounting (preserves full client lifecycle)
main_mcp.mount("api", api_server, as_proxy=True)

当挂载的服务器具有自定义寿命时，FastMCP 会自动使用代理挂载，但您可以使用`as_proxy`参数覆盖此行为。

​
### 与代理服务器的交互
当使用`FastMCP.as_proxy()`创建代理服务器时，挂载该服务器将始终使用代理挂载：

In [ ]:
# Create a proxy for a remote server
remote_proxy = FastMCP.as_proxy(Client("http://example.com/mcp"))

# Mount the proxy (always uses proxy mounting)
main_server.mount("remote", remote_proxy)

## 自定义分隔符
`mport_server() `和` mount() `都允许用户自定义组件前缀的分隔符。默认情况下，工具和提示使用 `_`，资源使用 `+`。

In [ ]:
await main_mcp.import_server(
    prefix="api",
    app=some_subserver,
    tool_separator="_",       # Tool name becomes: "api_sub_tool_name"
    resource_separator="+",   # Resource URI becomes: "api+data://sub_resource"
    prompt_separator="_"      # Prompt name becomes: "api_sub_prompt_name"
)

> 选择分隔符时要谨慎。某些 MCP 客户端（例如 Claude Desktop）可能对工具名称中允许的字符有限制（例如，`/`可能不支持）。默认值（_对于名称、+对于 URI）通常是安全的。